# Week 10 · Class 5 — Independent Capstone
## Historical Production Data Quality Pipeline

**Assessment notebook — not a guided teaching lesson.** Produce a self-contained, reproducible analytic report. Include your own explanations, assumptions, code outputs, and findings. This notebook must call and demonstrate the functions you implement in `src/production_pipeline.py`.

**Important:** Use the full project brief as the source of truth. Do not delete the original dataset or change input values.

## 1. Problem statement and objectives
Write 1–2 paragraphs describing why missing 6-hour observations, duplicates, incorrect measurements, and changing sensor reliability matter. State the required outputs and your policy for zeros versus missing readings.

Missing six-hour observations can hide production changes, while duplicate loads can count the same interval twice. Invalid or unreliable sensor readings can distort daily totals. This pipeline validates timestamps, keeps the latest duplicate load, aligns each well to a six-hour grid, and reports daily production and data-quality results. Missing production is kept as missing; a measured zero remains zero.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Find the project folder: it should contain src/ and data/.
PROJECT = Path.cwd()
if not (PROJECT / "src").exists():
    PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT))

from src.production_pipeline import (
    ingest_and_validate,
    repair_and_align,
    summarize_daily,
    run_pipeline,
)

INPUT = PROJECT / "data" / "historical_production_capstone.csv"
OUTPUT = PROJECT / "outputs"

print("Project folder:", PROJECT)
print("Input file exists:", INPUT.exists())

Project folder: c:\Users\muhib\Documents\Charlotte26\Module 2\Chalotte-week-10\day5_student_Capstone
Input file exists: True


## 2. Data ingestion and schema validation
Show raw row count, column names, data types, sample rows and invalid timestamps. Explain why timestamp parsing uses `errors="coerce"`.

In [2]:
raw = pd.read_csv(INPUT)

print("Raw rows:", len(raw))
print("Columns:", raw.columns.tolist())
display(raw.dtypes)
display(raw.head())

accepted, rejected = ingest_and_validate(INPUT)

print("Accepted rows:", len(accepted))
print("Rejected timestamp rows:", len(rejected))
display(rejected)

Raw rows: 47
Columns: ['well_id', 'timestamp', 'oil_bbl', 'water_bbl', 'gas_mscf', 'pressure_psi', 'status', 'ingested_at']


well_id             str
timestamp           str
oil_bbl         float64
water_bbl         int64
gas_mscf            str
pressure_psi        str
status              str
ingested_at         str
dtype: object

,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at
0,WELL-21,2026-04-06 18:00,77.0,22,151,2839,ACTIVE,2026-04-06 18:04
1,WELL-21,2026-04-08 00:00,68.0,24,161,2904,ACTIVE,2026-04-08 00:04
2,WELL-21,2026-04-09 00:00,68.0,22,169,2956,ACTIVE,2026-04-09 00:04
3,WELL-22,2026-04-06 00:00,78.0,24,152,2960,ACTIVE,2026-04-06 00:04
4,WELL-22,2026-04-07 00:00,78.0,25,160,not_available,ACTIVE,2026-04-07 00:04


Accepted rows: 46
Rejected timestamp rows: 1


,well_id,timestamp,oil_bbl,water_bbl,gas_mscf,pressure_psi,status,ingested_at,source_row_number,timestamp_raw,ingested_at_raw,rejection_reason
34,WELL-22,NaT,72.0,22,150,2960,ACTIVE,2026-04-10 01:00:00,36,2026-04-99 12:00,2026-04-10 01:00,Timestamp could not be parsed


## 3. Deduplication, alignment, and cleaning
Show how latest-ingestion duplicate precedence works. Explain how missing *timestamps* differ from missing *values*. Display a before-and-after example for a chosen well. Explain how you prevent false oil production totals.

In [3]:
cleaned, quality_report = repair_and_align(accepted)

print("Cleaned rows:", len(cleaned))
display(quality_report)

# Example: inspect WELL-21 before and after alignment.
well21_source = accepted.loc[accepted["well_id"] == "WELL-21"].copy()
well21_cleaned = cleaned.loc[cleaned["well_id"] == "WELL-21"].copy()

display(well21_source[["timestamp", "oil_bbl", "water_bbl", "gas_mscf"]])
display(
    well21_cleaned[
        ["timestamp", "is_inserted_slot", "oil_bbl", "water_bbl",
         "gas_mscf", "pressure_psi"]
    ]
)

print("Missing measurements in cleaned data:")
display(cleaned[["oil_bbl", "water_bbl", "gas_mscf", "pressure_psi"]].isna().sum())

Cleaned rows: 48


,metric,count
0,rows_after_timestamp_validation,46
1,duplicate_rows_removed,2
2,inserted_six_hour_slots,4
3,missing_oil_bbl_after_cleaning,6
4,missing_water_bbl_after_cleaning,5
5,missing_gas_mscf_after_cleaning,5
6,missing_pressure_psi_after_cleaning,0


,timestamp,oil_bbl,water_bbl,gas_mscf
0,2026-04-06 18:00:00,77.0,22,151
1,2026-04-08 00:00:00,68.0,24,161
2,2026-04-09 00:00:00,68.0,22,169
11,2026-04-06 00:00:00,68.0,22,145
12,2026-04-07 00:00:00,68.0,23,153
13,2026-04-08 06:00:00,71.0,-12,163
14,2026-04-09 06:00:00,71.0,23,171
22,2026-04-06 06:00:00,76.0,23,147
23,2026-04-06 12:00:00,NaN,24,149
24,2026-04-07 18:00:00,77.0,23,159


,timestamp,is_inserted_slot,oil_bbl,water_bbl,gas_mscf,pressure_psi
0,2026-04-06 00:00:00,False,68.0,22.0,145.0,2800.0
1,2026-04-06 06:00:00,False,76.0,23.0,147.0,2813.0
2,2026-04-06 12:00:00,False,NaN,24.0,149.0,2826.0
3,2026-04-06 18:00:00,False,77.0,22.0,151.0,2839.0
4,2026-04-07 00:00:00,False,68.0,23.0,153.0,2852.0
5,2026-04-07 06:00:00,True,NaN,NaN,NaN,2865.0
6,2026-04-07 12:00:00,False,74.0,22.0,157.0,2878.0
7,2026-04-07 18:00:00,False,77.0,23.0,159.0,2891.0
8,2026-04-08 00:00:00,False,68.0,24.0,161.0,2904.0
9,2026-04-08 06:00:00,False,71.0,NaN,163.0,2917.0


Missing measurements in cleaned data:


oil_bbl         6
water_bbl       5
gas_mscf        5
pressure_psi    0
dtype: int64

## 4. Daily resampling / aggregation
Report daily oil, water and gas totals, observation completeness, and incomplete days. Explain why sum(min_count=1) protects all-missing groups.

In [4]:
daily = summarize_daily(cleaned)

print("Daily summary rows:", len(daily))
display(daily)

incomplete_days = daily.loc[
    ~daily[
        ["oil_bbl_complete", "water_bbl_complete", "gas_mscf_complete"]
    ].all(axis=1)
]
print("Incomplete well-days:", len(incomplete_days))
display(incomplete_days)

Daily summary rows: 12


,well_id,date,expected_slots,valid_oil_readings,valid_water_readings,valid_gas_readings,oil_bbl,water_bbl,gas_mscf,oil_bbl_complete,water_bbl_complete,gas_mscf_complete
0,WELL-21,2026-04-06,4,3,4,4,221.0,91.0,592.0,False,True,True
1,WELL-21,2026-04-07,4,3,3,3,219.0,68.0,469.0,False,False,False
2,WELL-21,2026-04-08,4,4,3,4,290.0,71.0,656.0,True,False,True
3,WELL-21,2026-04-09,4,4,4,4,290.0,91.0,688.0,True,True,True
4,WELL-22,2026-04-06,4,4,4,4,330.0,99.0,620.0,True,True,True
5,WELL-22,2026-04-07,4,4,4,4,243.0,75.0,486.0,True,True,True
6,WELL-22,2026-04-08,4,2,3,3,159.0,76.0,512.0,False,False,False
7,WELL-22,2026-04-09,4,4,4,4,330.0,99.0,716.0,True,True,True
8,WELL-23,2026-04-06,4,3,3,3,273.0,81.0,483.0,False,False,False
9,WELL-23,2026-04-07,4,4,4,4,370.0,108.0,680.0,True,True,True


Incomplete well-days: 7


,well_id,date,expected_slots,valid_oil_readings,valid_water_readings,valid_gas_readings,oil_bbl,water_bbl,gas_mscf,oil_bbl_complete,water_bbl_complete,gas_mscf_complete
0,WELL-21,2026-04-06,4,3,4,4,221.0,91.0,592.0,False,True,True
1,WELL-21,2026-04-07,4,3,3,3,219.0,68.0,469.0,False,False,False
2,WELL-21,2026-04-08,4,4,3,4,290.0,71.0,656.0,True,False,True
6,WELL-22,2026-04-08,4,2,3,3,159.0,76.0,512.0,False,False,False
8,WELL-23,2026-04-06,4,3,3,3,273.0,81.0,483.0,False,False,False
10,WELL-23,2026-04-08,4,4,4,3,374.0,109.0,537.0,True,True,False
11,WELL-23,2026-04-09,4,3,3,3,282.0,81.0,561.0,False,False,False


## 5. Validation evidence and interpretation
Provide results of tests you wrote, expected-vs-actual checks for row counts, and at least three operational insights or limitations. Note that the data are synthetic.

In [5]:
run_pipeline(INPUT, OUTPUT)

expected_files = [
    "cleaned_production.csv",
    "data_quality_report.csv",
    "daily_production_summary.csv",
    "rejected_source_records.csv",
]

for name in expected_files:
    path = OUTPUT / name
    print(name, "exists:", path.exists())

cleaned_production.csv exists: True
data_quality_report.csv exists: True
daily_production_summary.csv exists: True
rejected_source_records.csv exists: True


In [6]:
checks = pd.DataFrame(
    [
        ("Raw source rows", len(raw), 47),
        ("Invalid timestamps rejected", len(rejected), 1),
        ("Final cleaned rows", len(cleaned), 48),
        ("Daily summary rows", len(daily), 12),
    ],
    columns=["Check", "Actual", "Target"],
)
checks["Pass"] = checks["Actual"] == checks["Target"]
display(checks)

,Check,Actual,Target,Pass
0,Raw source rows,47,47,True
1,Invalid timestamps rejected,1,1,True
2,Final cleaned rows,48,48,True
3,Daily summary rows,12,12,True


## 6. Conclusions and reproducibility
Explain your cleaning decisions, their impact on interpretation, and how another analyst can rerun the pipeline. Include a link to the GitHub Pull Request in your README (do not hard-code a private token).

Conclusions and reproducibility

This pipeline validates the required input columns and rejects rows with timestamps that cannot be parsed, recording a reason for each rejection. It keeps the latest ingested row for each well and timestamp, then aligns each well to a six-hour grid and marks inserted slots. Nonnumeric and negative measurements are treated as missing (`NaN`), while genuine zero readings are preserved. Production values are never interpolated; pressure is interpolated only across a single internal gap.

These choices affect how production is interpreted. Missing production is unknown, not zero, so daily totals use `sum(min_count=1)` and remain missing when there are no valid readings to sum. Completeness counts show when a daily total is based on fewer than the expected readings. The dataset is synthetic. The supplied timestamps use `M/D/YYYY H:MM`, while the brief specifies `YYYY-MM-DD HH:MM`; the parser handles the supplied format, and the malformed date is rejected.
